# 🧠 Day 01a: Neural Network Math — Build a Network From Scratch

---

## 🎯 What You'll Master Today
- Perceptron — the simplest neuron
- Multi-layer network forward pass
- Matrix multiplication view of neural nets
- Full forward pass in pure NumPy

**Goal:** Implement a forward pass from memory. Understand what PyTorch does under the hood.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  NEURAL NETWORK — It's Just Matrix Math                         ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Single Neuron:                                                  ║
║    output = activation(Σ(wᵢ · xᵢ) + b)                        ║
║           = activation(w·x + b)                                  ║
║                                                                   ║
║  Layer (matrix form):                                            ║
║    Z = X @ W + b        ← linear transform                     ║
║    A = activation(Z)    ← non-linearity                        ║
║                                                                   ║
║  Full Network:                                                   ║
║    X → [Linear→ReLU] → [Linear→ReLU] → [Linear→Sigmoid] → ŷ  ║
║         Layer 1           Layer 2           Output               ║
║                                                                   ║
║  Shape tracking:                                                  ║
║    X:     (batch, features)                                      ║
║    W1:    (features, hidden1)                                    ║
║    Z1:    (batch, hidden1)                                       ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np

# ============================================================
# 1. Single Perceptron — The Simplest Neuron
# ============================================================

def perceptron(x, w, b):
    """Single neuron: output = step(w·x + b)"""
    z = np.dot(w, x) + b      # linear combination
    return 1 if z >= 0 else 0  # step activation

# AND gate with perceptron
w_and = np.array([1, 1])
b_and = -1.5

print("AND gate (perceptron):")
for x1, x2 in [(0,0), (0,1), (1,0), (1,1)]:
    print(f"  ({x1}, {x2}) → {perceptron(np.array([x1, x2]), w_and, b_and)}")

# OR gate
w_or = np.array([1, 1])
b_or = -0.5

print("\nOR gate (perceptron):")
for x1, x2 in [(0,0), (0,1), (1,0), (1,1)]:
    print(f"  ({x1}, {x2}) → {perceptron(np.array([x1, x2]), w_or, b_or)}")

# XOR — impossible with single perceptron! (not linearly separable)
print("\n💡 XOR needs 2 layers — this is why we need DEEP networks!")

In [ ]:
# ============================================================
# 2. Activation Functions
# ============================================================

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def relu(z):
    return np.maximum(0, z)

def tanh(z):
    return np.tanh(z)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=-1, keepdims=True))  # numerical stability
    return exp_z / exp_z.sum(axis=-1, keepdims=True)

z = np.array([-2, -1, 0, 1, 2])

print(f"z:        {z}")
print(f"sigmoid:  {sigmoid(z).round(4)}   ← squash to (0,1)")
print(f"relu:     {relu(z)}       ← kill negatives")
print(f"tanh:     {tanh(z).round(4)}  ← squash to (-1,1)")
print(f"softmax:  {softmax(z).round(4)}  ← probabilities (sum=1)")

print("\n💡 Hidden layers: ReLU (default). Output: sigmoid (binary), softmax (multi-class)")

In [ ]:
# ============================================================
# 3. Layer as Matrix Multiplication
# ============================================================

# Input: batch of 3 samples, 4 features each
X = np.random.randn(3, 4)

# Layer 1: 4 inputs → 5 hidden neurons
W1 = np.random.randn(4, 5) * 0.01  # (features, hidden)
b1 = np.zeros(5)                     # (hidden,)

Z1 = X @ W1 + b1     # (3, 4) @ (4, 5) = (3, 5)
A1 = relu(Z1)         # same shape

print(f"Input X:  {X.shape}")
print(f"W1:       {W1.shape}")
print(f"Z1 = X@W1+b1: {Z1.shape}")
print(f"A1 = relu(Z1): {A1.shape}")

# Layer 2: 5 hidden → 1 output
W2 = np.random.randn(5, 1) * 0.01
b2 = np.zeros(1)

Z2 = A1 @ W2 + b2     # (3, 5) @ (5, 1) = (3, 1)
A2 = sigmoid(Z2)       # final output

print(f"\nW2:       {W2.shape}")
print(f"Z2 = A1@W2+b2: {Z2.shape}")
print(f"Output:   {A2.flatten().round(4)}")

print("\n💡 Shape rule: (batch, in) @ (in, out) + (out,) → (batch, out)")

In [ ]:
# ============================================================
# 4. Full Neural Network Forward Pass — From Scratch
# ============================================================

class NeuralNetwork:
    """Simple feed-forward network: 2 hidden layers + output."""
    
    def __init__(self, input_dim, hidden1, hidden2, output_dim):
        # Xavier initialization: scale by sqrt(1/fan_in)
        self.W1 = np.random.randn(input_dim, hidden1) * np.sqrt(1 / input_dim)
        self.b1 = np.zeros(hidden1)
        self.W2 = np.random.randn(hidden1, hidden2) * np.sqrt(1 / hidden1)
        self.b2 = np.zeros(hidden2)
        self.W3 = np.random.randn(hidden2, output_dim) * np.sqrt(1 / hidden2)
        self.b3 = np.zeros(output_dim)
    
    def forward(self, X):
        """Forward pass: X → hidden1 → hidden2 → output"""
        # Layer 1
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = relu(self.Z1)
        
        # Layer 2
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = relu(self.Z2)
        
        # Output layer
        self.Z3 = self.A2 @ self.W3 + self.b3
        self.A3 = sigmoid(self.Z3)
        
        return self.A3
    
    def predict(self, X):
        return (self.forward(X) >= 0.5).astype(int)

# Test it
nn = NeuralNetwork(input_dim=4, hidden1=8, hidden2=4, output_dim=1)
X_test = np.random.randn(5, 4)
output = nn.forward(X_test)

print(f"Network architecture: 4 → 8 → 4 → 1")
print(f"Input:  {X_test.shape}")
print(f"Output: {output.flatten().round(4)}")
print(f"Predictions: {nn.predict(X_test).flatten()}")

# Count parameters
total_params = (4*8 + 8) + (8*4 + 4) + (4*1 + 1)
print(f"\nTotal parameters: {total_params}")
print(f"  Layer 1: {4*8} weights + {8} biases = {4*8+8}")
print(f"  Layer 2: {8*4} weights + {4} biases = {8*4+4}")
print(f"  Layer 3: {4*1} weights + {1} bias = {4*1+1}")

In [ ]:
# ============================================================
# 5. XOR Problem — Why We Need Hidden Layers
# ============================================================

# XOR truth table — not linearly separable
X_xor = np.array([[0,0], [0,1], [1,0], [1,1]])
y_xor = np.array([[0], [1], [1], [0]])

# Manual XOR network (hand-crafted weights)
# Hidden layer: 2 neurons that detect (AND, OR)
W1_xor = np.array([[1, 1],    # neuron1 weights
                    [1, 1]])   # neuron2 weights
b1_xor = np.array([-1.5, -0.5])  # AND threshold, OR threshold

# Output layer: combines to get XOR = OR AND (NOT AND)
W2_xor = np.array([[-1],    # negate AND
                    [1]])    # keep OR
b2_xor = np.array([-0.5])

# Forward pass
Z1 = X_xor @ W1_xor + b1_xor
A1 = (Z1 >= 0).astype(float)  # step activation
Z2 = A1 @ W2_xor + b2_xor
A2 = (Z2 >= 0).astype(float)

print("XOR Network (hand-crafted):")
print(f"  Input → Hidden (AND, OR) → Output (XOR)")
for i in range(4):
    print(f"  {X_xor[i]} → hidden={A1[i].astype(int)} → output={int(A2[i][0])} (expected={y_xor[i][0]})")

print("\n💡 XOR = OR AND (NOT AND) — needs at least 1 hidden layer!")

---

## 🗺️ Neural Network Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  NEURAL NETWORK MATH                                            ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  FORWARD PASS:                                                   ║
║    Z = X @ W + b          (linear)                              ║
║    A = activation(Z)      (non-linear)                          ║
║    Repeat for each layer                                         ║
║                                                                   ║
║  SHAPES:                                                         ║
║    X:  (batch, in_features)                                     ║
║    W:  (in_features, out_features)                              ║
║    b:  (out_features,)                                          ║
║    Z:  (batch, out_features)                                    ║
║                                                                   ║
║  PARAMS: in×out + out per layer                                 ║
║                                                                   ║
║  INIT: Xavier = rand * sqrt(1/fan_in)                           ║
║        He     = rand * sqrt(2/fan_in)  ← for ReLU              ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What does a single neuron compute? | z = w·x + b, then activation(z). It's a linear transform + non-linearity |
| 2 | Why do we need activation functions? | Without them, stacking layers = one big linear transform. Non-linearity enables learning complex patterns |
| 3 | Why can't a perceptron learn XOR? | XOR isn't linearly separable. Need hidden layer to create intermediate representations |
| 4 | Xavier vs He initialization? | Xavier: sqrt(1/fan_in) for sigmoid/tanh. He: sqrt(2/fan_in) for ReLU. Prevents vanishing/exploding gradients |
| 5 | How to count parameters? | Each layer: (in × out) weights + out biases. Sum across all layers |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Neuron = w·x + b → activation                       │
## │  • Layer = X @ W + b → activation (matrix form)        │
## │  • ReLU for hidden, sigmoid/softmax for output         │
## │  • Params per layer: in×out + out                      │
## │  • XOR proves need for depth                           │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Backpropagation** — how networks actually learn